In [ ]:
###--- BLAST ---###

import pandas as pd
from Bio import SeqIO
import os

base_path = "./"
blast_file = os.path.join(base_path, "./blast_results.txt")
query_fasta = os.path.join(base_path, "./combined_query.fasta")
output_csv = os.path.join(base_path, "./filtered_blast_results_ID80.csv")

# bring the entire query length 
print("loading query length..")
query_len_dict = {}
for record in SeqIO.parse(query_fasta, "fasta"):
    query_len_dict[record.id] = len(record.seq)

# read BLAST result 
cols = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", 
        "qstart", "qend", "sstart", "send", "evalue", "bitscore"]
df = pd.read_csv(blast_file, sep="\t", names=cols)

# calculate the coverage (Aligned length/Query length * 100)
df['qlen'] = df['qseqid'].map(query_len_dict) 
df['coverage'] = (df['length'] / df['qlen']) * 100

# Filtering (Identity >= 80, Coverage >= 70, E-value < 1e-5)
filtered_df = df[
    (df['pident'] >= 80) & 
    (df['coverage'] >= 70) & 
    (df['evalue'] < 1e-5)
].copy()

# sort and print
final_columns = ['qseqid', 'sseqid', 'pident', 'coverage', 'length', 'qlen', 'evalue', 'bitscore']
filtered_df = filtered_df[final_columns]
filtered_df = filtered_df.sort_values(by="coverage", ascending=False)

print("-" * 60)
print(f"{len(df)}: {len(filtered_df)} filtered.")
print("-" * 60)
print(filtered_df.head())

filtered_df.to_csv(output_csv, index=False)

In [ ]:
###---Kind of read me---###

#qseqid: query sequence id
#sseqid: matched locus id within reference genome file
#pident: percent of identity(%)
#length: matched length
#qlen: query length

In [ ]:
###--- Orthologs extraction ---###

import pandas as pd
import os
import re

base_path = "./"
output_dir = "./Orthologs/genotype1"
os.makedirs(output_dir, exist_ok=True)

blast_csv = os.path.join(base_path, "./genotype1/genotype1_filtered_blast_results_ID80.csv")
ortholog_file = os.path.join(base_path, "./N3.tsv")
output_csv = os.path.join(output_dir, "genotype1_orthologs_extracted.csv")

print("loading the data..")
df_blast = pd.read_csv(blast_csv)

# read Ortholog data.tsv
try:
    
    df_ortho = pd.read_csv(ortholog_file, sep='\t', dtype=str, quoting=3).fillna("")
    print(f"Ortholog data loaded: {df_ortho.shape}")
except FileNotFoundError:
    print("No file found.")
    exit()

# Select genotypes what I used for the experiment
target_map = {
    'genome_file.noctgs': 'genotype1',
}

# set the standard column (genotype of interest, GOI) to map the contents in corresponding row
ANCHOR_COL = 'genotype1.flt.noctgs' 

print("Ortholog map generating (std: select the genotype)") # genotype can be one 
GOI_to_row_idx = {}

if ANCHOR_COL in df_ortho.columns:
    # take the row number if the GOI ID presents
    for idx, row in df_ortho.iterrows():
        cell_value = row[ANCHOR_COL]
        if cell_value:
            ids = [x.strip() for x in str(cell_value).split(',') if x.strip()]
            for gene_id in ids:
                GOI_to_row_idx[gene_id] = idx
else:
    print(f"{ANCHOR_COL} is not found in Ortholog data")
    exit()

print(f"Mapping completes: {len(GOI_to_row_idx)} GOI locus obtained")

# Name of corresponding locus calling in different genotypes
print("Ortholog name calling..")

df_blast['sseqid_clean'] = df_blast['sseqid'].astype(str).str.strip()
result_cols = {col: [] for col in target_map.values()}

def get_ortholog_name(sseqid, target_tsv_col):
    # 1. sseqid(GOI ID) where
    row_idx = GOI_to_row_idx.get(sseqid)
    
    if row_idx is None:
        return "absent" # no matched ID to GOI ID
    
    # 2. matched genotype calling
    if target_tsv_col not in df_ortho.columns:
        return "N/A" # no genotype exists
        
    ortholog_name = df_ortho.at[row_idx, target_tsv_col]
    
    # 3. absent = no match
    if not ortholog_name or str(ortholog_name).strip() == "":
        return "absent"
    else:
        return str(ortholog_name).strip()

# Apply the function to find existence
for tsv_col, out_name in target_map.items():
    print(f" - Searching in {out_name}...")
    df_blast[out_name] = df_blast['sseqid_clean'].apply(
        lambda x: get_ortholog_name(x, tsv_col)
    )

# Save the result
final_cols = ['qseqid', 'sseqid'] + list(target_map.values())
final_df = df_blast[final_cols].copy()

print("-" * 60)
print("analysis completes")
print(final_df.head())
print("-" * 60)

matched_counts = final_df[list(target_map.values())].apply(lambda x: (x != 'absent').sum())
print("Number of Ortholog in each genotype:")
print(matched_counts)

final_df.to_csv(output_csv, index=False)
print(f"file saved in: {output_csv}")

In [ ]:
###--- Generate ortholog presence/absent table ---###

import pandas as pd
import os

base_path = "./"
input_csv = os.path.join(base_path, "./Orthologs/genotype1/genotype1_orthologs_extracted.csv")
output_csv = os.path.join(base_path, "./Orthologs/genotype1/genotype1_ortholog_presence.csv")

print("loading the data..")
try:
    df = pd.read_csv(input_csv)
    print(f"data loaded: {len(df)} rows")
except FileNotFoundError:
    print("File not found")
    exit()

# Select the columns 
target_columns = [
    'genotype1', 'genotype2', 'genotype3'
]

# convert value1 to value2
def convert_to_presence(value):
    if pd.isna(value):
        return 'absent'
    val_str = str(value).strip().lower() # to prevent false loading
    
    # 'absent' -> 'absent'
    if val_str in ['absent', '']:
        return 'absent'
    else:
        # else -> 'present'
        return 'present'

print("Data refining..")

# Apply function to all columns
for col in target_columns:
    if col in df.columns:
        df[col] = df[col].apply(convert_to_presence)
    else:
        print(f"'{col}' does not exist.")

# Save the result
print("-" * 60)
print("conversion completes, head")
print(df[['qseqid', 'sseqid'] + target_columns[:3]].head()) 
print("-" * 60)

present_counts = df[target_columns].apply(lambda x: (x == 'present').sum())
print("Number of present in each genotype:")
print(present_counts)

df.to_csv(output_csv, index=False)
print(f"file saved in: {output_csv}")